# Homework 4 

In [7]:
!pip install -U minsearch qdrant_client

In [1]:
import requests
import pandas as pd

url_prefix = 'https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/03-evaluation/'
docs_url = url_prefix + 'search_evaluation/documents-with-ids.json'
documents = requests.get(docs_url).json()

ground_truth_url = url_prefix + 'search_evaluation/ground-truth-data.csv'
df_ground_truth = pd.read_csv(ground_truth_url)
ground_truth = df_ground_truth.to_dict(orient='records')

In [2]:
from tqdm.auto import tqdm

def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if True in line:
            cnt = cnt + 1

    return cnt / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score = total_score + 1 / (rank + 1)

    return total_score / len(relevance_total)

def evaluate(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        doc_id = q['document']
        results = search_function(q)
        relevance = [d['id'] == doc_id for d in results]
        relevance_total.append(relevance)

    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mrr(relevance_total),
    }

## Question 1. Minisearch text

In [11]:
import minsearch

index = minsearch.Index(
    text_fields=["question", "section", "text"],
    keyword_fields=["course", "id"]
)

index.fit(documents)

In [13]:
def minsearch_search(query, course):
    #boost = {'question': 3.0, 'section': 0.5}
    
    boost = {'question': 1.5, 'section': 0.1}

    results = index.search(
        query=query,
        filter_dict={'course': course},
        boost_dict=boost,
        num_results=5
    )

    return results

In [14]:
relevance_total = []

for q in tqdm(ground_truth):
    doc_id = q['document']
    results = minsearch_search(query=q['question'], course=q['course'])
    relevance = [d['id'] == doc_id for d in results]
    relevance_total.append(relevance)

  0%|          | 0/4627 [00:00<?, ?it/s]

In [15]:
hit_rate(relevance_total), mrr(relevance_total)


(0.848714069591528, 0.7288235717887772)

In [16]:
print(f"The hit rate is {hit_rate(relevance_total):.2f}")


The hit rate is 0.85


### Embeddings

In [17]:
from minsearch import VectorSearch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline

In [18]:
texts = []

for doc in documents:
    t = doc['question']
    texts.append(t)

pipeline = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)
X = pipeline.fit_transform(texts)

## Q2. Vector search for question

In [19]:
vindex = VectorSearch(keyword_fields={'course'})
vindex.fit(X, documents)

In [24]:
def minsearch_search(query, course):
    #boost = {'question': 3.0, 'section': 0.5}
    
    boost = {'question': 1.5, 'section': 0.1}
    query_vec = pipeline.transform([query])

    results =vindex.search(
        query_vec,
        filter_dict={'course': course},
        num_results=5
    )

    return results

In [25]:
relevance_total = []

for q in tqdm(ground_truth):
    doc_id = q['document']
    results = minsearch_search(query=q['question'], course=q['course'])
    relevance = [d['id'] == doc_id for d in results]
    relevance_total.append(relevance)

  0%|          | 0/4627 [00:00<?, ?it/s]

In [26]:
print(f"The mrr rate is {mrr(relevance_total):.2f}")


The mrr rate is 0.36


## Q3. Vector search for question and answer


In [28]:
texts = []

for doc in documents:
    t = doc['question'] + ' ' + doc['text']
    texts.append(t)

In [29]:
pipeline = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)
X = pipeline.fit_transform(texts)

In [30]:
vindex = VectorSearch(keyword_fields={'course'})
vindex.fit(X, documents)

In [31]:
def minsearch_search(query, course):
    #boost = {'question': 3.0, 'section': 0.5}
    
    boost = {'question': 1.5, 'section': 0.1}
    query_vec = pipeline.transform([query])

    results =vindex.search(
        query_vec,
        filter_dict={'course': course},
        num_results=5
    )

    return results

In [32]:
relevance_total = []

for q in tqdm(ground_truth):
    doc_id = q['document']
    results = minsearch_search(query=q['question'], course=q['course'])
    relevance = [d['id'] == doc_id for d in results]
    relevance_total.append(relevance)

  0%|          | 0/4627 [00:00<?, ?it/s]

In [33]:
print(f"The hit rate is {hit_rate(relevance_total):.2f}")

The hit rate is 0.82


## Q4. Qdrant
Now let's evaluate the following settings in Qdrant:

In [35]:
from fastembed import TextEmbedding

In [36]:
text = doc['question'] + ' ' + doc['text']
model_handle = "jinaai/jina-embeddings-v2-small-en"
limit = 5
embedding_model = TextEmbedding(model_name=model_handle)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

c:\Users\mikes\anaconda3\envs\py11\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mikes\AppData\Local\Temp\fastembed_cache\models--xenova--jina-embeddings-v2-small-en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Fallin

tokenizer_config.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.onnx:   0%|          | 0.00/130M [00:00<?, ?B/s]

In [45]:
from qdrant_client import QdrantClient, models
import requests

# Connect to Qdrant (make sure it's running first!)
client = QdrantClient("http://localhost:6333")

collection_name = "ml-zoomcamp-qa"

In [51]:
# (Re)create Qdrant collection and insert documents with embeddings
from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding

client = QdrantClient("http://localhost:6333")
collection_name = "ml-zoomcamp-qa"

# Get embedding size for the model
embedding_model = TextEmbedding(model_name="jinaai/jina-embeddings-v2-small-en")
test_vec = list(embedding_model.embed(["test text"]))[0]
vector_size = len(test_vec)

# Recreate collection (deletes if exists, then creates)
client.recreate_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=vector_size,
        distance=models.Distance.COSINE
    )
)

# Prepare and insert points
payloads = []
vectors = []
for doc in documents:
    text = doc['question'] + ' ' + doc['text']
    vector = list(embedding_model.embed([text]))[0]
    vectors.append(vector)
    payloads.append({
        "id": doc["id"],
        "course": doc["course"],
        "question": doc["question"],
        "text": doc["text"]
    })

client.upsert(
    collection_name=collection_name,
    points=[
        models.PointStruct(
            id=idx,
            vector=vec,
            payload=payload
        )
        for idx, (vec, payload) in enumerate(zip(vectors, payloads))
    ]
)

C:\Users\mikes\AppData\Local\Temp\ipykernel_15548\575052226.py:14: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [47]:
def qdrant_search(query, course, embedding_model, qdrant_client, collection_name="ml-zoomcamp-qa", limit=5):
    # Embed the query (question + answer if needed)
    query_text = query  # or: query_text = q['question'] + ' ' + q['text']
    #query_vector = list(embedding_model.embed([query_text])[0])
    query_vector = list(embedding_model.embed([query_text]))[0]

    # Build filter for course
    course_filter = {
        "must": [
            {
                "key": "course",
                "match": {"value": course}
            }
        ]
    }

    # Search in Qdrant
    results = qdrant_client.search(
        collection_name=collection_name,
        query_vector=query_vector,
        limit=limit,
        query_filter=course_filter
    )

    # Convert Qdrant results to your expected format (list of dicts with 'id')
    return [{"id": point.payload["id"]} for point in results]

In [52]:
relevance_total = []

for q in tqdm(ground_truth):
    doc_id = q['document']
    # Example call inside your evaluation loop
    results = qdrant_search(
        query=q['question'],
        course=q['course'],
        embedding_model=embedding_model,
        qdrant_client=client,
        collection_name=collection_name
    )
    relevance = [d['id'] == doc_id for d in results]
    relevance_total.append(relevance)

  0%|          | 0/4627 [00:00<?, ?it/s]

C:\Users\mikes\AppData\Local\Temp\ipykernel_15548\3893112036.py:18: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = qdrant_client.search(


In [53]:
print(f"The MRR for qdrant is {mrr(relevance_total):.2f}")

The MRR for qdrant is 0.85


## Q5 Cosine simiarity

In [56]:
import numpy as np

In [57]:
def normalize(u):
    norm = np.sqrt(u.dot(u))
    return u / norm

def cosine(u, v):
    u = normalize(u)
    v = normalize(v)
    return u.dot(v)

In [58]:
results_url = url_prefix + 'rag_evaluation/data/results-gpt4o-mini.csv'
df_results = pd.read_csv(results_url)

In [59]:
pipeline = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)

In [60]:
pipeline.fit(df_results.answer_llm + ' ' + df_results.answer_orig + ' ' + df_results.question)

Pipeline(steps=[('tfidfvectorizer', TfidfVectorizer(min_df=3)),
                ('truncatedsvd',
                 TruncatedSVD(n_components=128, random_state=1))])

In [68]:
print(v_llm.shape, v_orig.shape)
len(df_results.answer_llm)

(1, 128) (1, 128)


1830

In [70]:
v_llm = np.zeros((len(df_results.answer_llm), 128) )
v_orig = np.zeros((len(df_results.answer_llm), 128) )
cosine_similarity = np.array(np.zeros((len(df_results.answer_llm), 1)))
for ind in range (len(df_results)):
    v_llm[ind], v_orig[ind] = pipeline.transform([df_results.answer_llm[ind]]), pipeline.transform([df_results.answer_orig[ind]])
    cosine_similarity[ind] = cosine(v_llm[ind], v_orig[ind])


print(f"Cosine similarity: {np.mean(cosine_similarity):.4f}")

Cosine similarity: 0.8416


## Q6. Rouge


In [71]:
!pip install rouge

In [74]:
from rouge import Rouge
rouge_scorer = Rouge()

r = df_results.iloc[10]
scores = rouge_scorer.get_scores(r.answer_llm, r.answer_orig)[0]
scores["rouge-1"]

{'r': 0.45454545454545453, 'p': 0.45454545454545453, 'f': 0.45454544954545456}

In [75]:
scores

{'rouge-1': {'r': 0.45454545454545453,
  'p': 0.45454545454545453,
  'f': 0.45454544954545456},
 'rouge-2': {'r': 0.21621621621621623,
  'p': 0.21621621621621623,
  'f': 0.21621621121621637},
 'rouge-l': {'r': 0.3939393939393939,
  'p': 0.3939393939393939,
  'f': 0.393939388939394}}

In [76]:
f1_score_avg = 0
for ind in range(len(df_results)):
    r = df_results.iloc[ind]

    scores = rouge_scorer.get_scores(r.answer_llm, r.answer_orig)[0]
    f1_score_avg = f1_score_avg + scores['rouge-1']['f']

print(f"F1 score average: {f1_score_avg / len(df_results):.4f}")



F1 score average: 0.3517
